# Natural Language Autoencoders: explain and reconstruct an activation

A Natural Language Autoencoder (NLA) is a pair of fine-tuned language models trained to send a residual-stream activation through natural language and back. The **activation verbalizer** (AV) reads one activation, injected as a single input embedding, and writes an explanation. The **activation reconstructor** (AR) reads that explanation and predicts the original activation. The two were trained together, so the reconstruction quality measures how much of the vector survives the trip through text.

This notebook runs that round trip with `TransformerBridge` hooks and generation, without a model server. The mechanics section below executes the same helpers on a tiny Qwen2 checkpoint and checks them against plain Hugging Face. It demonstrates the *inference protocol* of the released models; it is not a replication of the paper's training or evaluation.

Reconstruction fidelity shows that a vector is preserved, not that the explanation is faithful to what a human would read into it. A reconstruction can be accurate while the text still misleads.

The method and the released models are described in [Natural Language Autoencoders Produce Unsupervised Explanations of LLM Activations](https://transformer-circuits.pub/2026/nla/). The authors' [reference implementation](https://github.com/kitft/natural_language_autoencoders) serves the AV through SGLang embedding injection; this notebook does not import or depend on it.

## The round trip

| Step | Model | Input | Output |
|---|---|---|---|
| Extract | target model | a prompt and one token position | the raw output of block $L$ at that position, a vector $h \in \mathbb{R}^{d}$ |
| Verbalize | AV | a fixed chat prompt with one placeholder token | an explanation inside `<explanation>...</explanation>` |
| Reconstruct | AR | the explanation text, in a fixed raw template | a vector $\hat h \in \mathbb{R}^{d}$ |
| Score | none | $h$ and $\hat h$ | cosine similarity and direction-normalized MSE |

The AV is a full copy of the target architecture. Before its forward pass, the embedding row of the placeholder token is replaced by $h$, rescaled to a fixed norm recorded with the artifact. The AR is the target architecture truncated to its first $L + 1$ blocks, followed by a separately released bias-free linear *value head* applied to the final token's block-$L$ output. Its final norm and LM head are never used.

The [Patchscopes demo](https://github.com/TransformerLensOrg/TransformerLens/blob/dev/demos/Patchscopes_Generation_Demo.ipynb) also patches an activation into a generating model, but uses an untrained inspection prompt and has no reconstruction step.

## Requirements

The mechanics section runs on CPU in seconds and downloads only a tiny Qwen2 checkpoint. Every helper is defined in this notebook on top of public `TransformerBridge` methods: `run_with_cache`, `hooks`, and `generate`.

In [1]:
# NBVAL_IGNORE_OUTPUT
import gc
import math
import os
import re

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import torch
from huggingface_hub.utils import logging as hf_hub_logging
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

from transformer_lens.model_bridge import TransformerBridge

# Loading messages and progress bars would otherwise land in the checked cell outputs.
hf_hub_logging.set_verbosity_error()
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

DEVICE = "cpu"  # Stable saved outputs and portable notebook validation.
EXPLANATION_OPEN, EXPLANATION_CLOSE = "<explanation>", "</explanation>"
pd.set_option("display.max_colwidth", None)
_ = torch.manual_seed(0)

## Reading the artifact sidecar

Each released AV and AR repository ships an `nla_meta.yaml` sidecar that records the contract the models were trained with: the extraction layer, the width, the placeholder character and its neighboring token ids, the injection norm, the prompt templates, and the AR's expected prompt suffix. Every later step reads these values instead of hardcoding them, so `check_sidecar` validates them against the live tokenizer first. A tokenizer that splits the placeholder differently, or a template that drifted, would otherwise produce plausible-looking text from the wrong input.

The reconstruction metric is direction-normalized: both vectors are rescaled to norm $\sqrt{d}$ before comparison. The sidecar records this as `mse_scale`, which is checked to equal $\sqrt{d}$ to within its serialized precision.

In [2]:
def check_sidecar(meta, role, tokenizer, d_model, layer):
    # Validate a sidecar against its role and the live tokenizer, and return its key values.
    assert meta["kind"] == "nla_model" and meta["schema_version"] == 2, "unknown sidecar schema"
    assert meta["role"] == role, f"expected a {role} sidecar, got {meta['role']}"
    assert meta["d_model"] == d_model and meta["extraction_layer_index"] == layer
    extraction, tokens, templates = meta["extraction"], meta["tokens"], meta["prompt_templates"]
    assert math.isclose(extraction["mse_scale"], math.sqrt(d_model), rel_tol=1e-12)

    char = tokens["injection_char"]
    char_ids = tokenizer.encode(char, add_special_tokens=False)
    assert char_ids == [tokens["injection_token_id"]], "placeholder is not one recorded token"
    assert char_ids[0] != tokenizer.unk_token_id, "placeholder encodes to the unknown token"
    assert templates["av"].count("{injection_char}") == 1
    assert templates["ar"].count("{explanation}") == 1

    if role == "av":
        norm = extraction["injection_scale"]
        assert isinstance(norm, (int, float)) and math.isfinite(norm) and norm > 0
        assert tokens["critic_suffix_ids"] is None
    else:
        assert extraction["injection_scale"] is None
        probe = tokenizer.encode(templates["ar"].format(explanation="x"), add_special_tokens=True)
        suffix = tokens["critic_suffix_ids"]
        assert probe[-len(suffix) :] == suffix, "reconstructor prompt suffix drifted"
    return meta

## Extracting an activation

`blocks.{L}.hook_out` is the raw residual stream after zero-based block $L$, the same tensor as Hugging Face's `hidden_states[L + 1]`. The Bridge must be booted without compatibility mode so that this is the basis the NLA was trained on. Only that one hook is cached, and the selected row is copied to the CPU in float32 so the target model can be released before the AV is loaded.

In [3]:
def extract(bridge, tokens, layer, position=-1):
    # Return the detached block-`layer` output at one token position, and the resolved position.
    assert not bridge.compatibility_mode, "NLAs read the raw residual basis; boot a raw Bridge"
    assert tokens.ndim == 2 and tokens.shape[0] == 1 and tokens.shape[1] > 0, "one unpadded row"
    seq_len = tokens.shape[1]
    assert -seq_len <= position < seq_len, f"position {position} outside {seq_len} tokens"
    hook_name = f"blocks.{layer}.hook_out"
    with torch.no_grad():
        _, cache = bridge.run_with_cache(tokens, names_filter=hook_name)
    vector = cache[hook_name][0, position].detach().to("cpu", torch.float32).clone()
    del cache
    return vector, position % seq_len

## Injecting it into the verbalizer

The AV prompt contains the placeholder character exactly once. `make_injection_hook` rescales the activation to the sidecar's injection norm in float64, casts it to the embedding's dtype, and overwrites only the placeholder row at `hook_embed`.

Generation needs care. With the KV cache, the first forward pass (the prefill) covers the whole prompt and later passes see only the newest token, so the overwrite applies on the first call only. Without the cache, every pass recomputes the full prefix from position 0, so the overwrite applies on every call. The hook records each call's sequence length and whether it injected, so the mechanics checks below can confirm both patterns. A new hook is built for every generation, so no state carries over between calls.

In [4]:
def make_injection_hook(vector, position, norm, use_past_kv_cache):
    # Build a hook_embed hook that writes the rescaled vector at the placeholder position.
    vector = vector.detach().to("cpu", torch.float64)
    vector_norm = vector.norm()
    assert torch.isfinite(vector).all() and vector_norm > 0, "need a finite nonzero vector"
    scaled = vector / vector_norm * norm
    calls = []

    def hook(embeddings, hook):
        inject = not (use_past_kv_cache and calls)
        calls.append((embeddings.shape[1], inject))
        if not inject:
            return embeddings
        assert embeddings.shape[1] > position, "the forward pass does not cover the placeholder"
        embeddings = embeddings.clone()
        embeddings[:, position] = scaled.to(embeddings.device, embeddings.dtype)
        return embeddings

    return hook, calls

## Verbalizing

`render_av_prompt` applies the AV's chat template once, with the generation prompt appended, and locates the single placeholder token. It checks the neighboring token ids against the sidecar, which catches a template or tokenizer change that would shift the injection site. `verbalize` then samples a continuation inside `bridge.hooks(...)`, which removes the injection hook on exit, including when generation raises. Sampling settings are explicit and bounded; pass `do_sample=False` for greedy decoding.

In [5]:
def render_av_prompt(tokenizer, meta):
    # Tokenize the AV chat prompt once and return its ids and the placeholder position.
    tokens = meta["tokens"]
    content = meta["prompt_templates"]["av"].format(injection_char=tokens["injection_char"])
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": content}], tokenize=True, add_generation_prompt=True
    )
    ids = list(rendered["input_ids"] if hasattr(rendered, "keys") else rendered)
    matches = [i for i, token in enumerate(ids) if token == tokens["injection_token_id"]]
    assert len(matches) == 1, "the AV prompt must contain the placeholder exactly once"
    position = matches[0]
    assert 0 < position < len(ids) - 1, "the placeholder needs a token on each side"
    assert ids[position - 1] == tokens["injection_left_neighbor_id"], "left neighbor drifted"
    assert ids[position + 1] == tokens["injection_right_neighbor_id"], "right neighbor drifted"
    return torch.tensor([ids]), position


def verbalize(
    av,
    tokenizer,
    meta,
    vector,
    *,
    max_new_tokens=256,
    do_sample=True,
    temperature=1.0,
    use_past_kv_cache=True,
    seed=0,
):
    # Generate an explanation of `vector` and return the raw continuation and the parse result.
    assert not av.compatibility_mode, "the AV must run on raw weights"
    assert 0 < max_new_tokens and (not do_sample or temperature > 0)
    prompt, position = render_av_prompt(tokenizer, meta)
    assert prompt.shape[1] + max_new_tokens <= av.cfg.n_ctx, "prompt plus budget exceeds n_ctx"
    norm = meta["extraction"]["injection_scale"]
    hook, calls = make_injection_hook(vector, position, norm, use_past_kv_cache)
    torch.manual_seed(seed)
    with torch.no_grad(), av.hooks(fwd_hooks=[("hook_embed", hook)]):
        output = av.generate(
            prompt.to(av.cfg.device),
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            use_past_kv_cache=use_past_kv_cache,
            return_type="tokens",
            verbose=False,
        )
    continuation = output[0, prompt.shape[1] :].tolist()
    raw_text = tokenizer.decode(continuation, skip_special_tokens=True)
    text, status = parse_explanation(raw_text)
    return {
        "raw_text": raw_text,
        "token_ids": continuation,
        "text": text,
        "status": status,
        "hook_calls": calls,
    }

## Parsing the explanation

The AV is trained to wrap its explanation in `<explanation>...</explanation>`. `parse_explanation` accepts the stripped inner text of exactly one complete region and reports every other outcome as a named failure: `missing`, `multiple`, `nested`, `incomplete`, or `empty`. A failure returns `None` rather than falling back to the raw continuation, because reconstructing arbitrary model output would report a score for text the AR was never trained on. Failures are results and are reported, not dropped.

This is stricter than the authors' reference client, which takes the first `<explanation>` region it finds and, when there is none, returns the raw continuation with a warning. Parse-failure counts from the two are therefore not directly comparable.

In [6]:
def parse_explanation(text):
    # Return (inner text, "ok") for exactly one complete region, otherwise (None, reason).
    tags = sorted(
        [(m.start(), +1) for m in re.finditer(re.escape(EXPLANATION_OPEN), text)]
        + [(m.start(), -1) for m in re.finditer(re.escape(EXPLANATION_CLOSE), text)]
    )
    if not tags:
        return None, "missing"
    depth, regions, nested = 0, 0, False
    for _, step in tags:
        depth += step
        if depth < 0:
            return None, "incomplete"
        nested |= depth > 1
        regions += step < 0 and depth == 0
    if depth != 0:
        return None, "incomplete"
    if nested:
        return None, "nested"
    if regions > 1:
        return None, "multiple"
    start = text.index(EXPLANATION_OPEN) + len(EXPLANATION_OPEN)
    inner = text[start : text.index(EXPLANATION_CLOSE)].strip()
    return (inner, "ok") if inner else (None, "empty")

## Reconstructing

The AR reads the explanation in a fixed raw template, not a chat prompt, tokenized once with the tokenizer's own special-token policy. `reconstruct` checks that the rendered prompt ends with the sidecar's recorded suffix, reads the final token's block-$L$ output with `extract`, and applies the value head.

The AR checkpoint has only $L + 1$ blocks. Its final norm and LM head are absent from the released weights, so Hugging Face initializes them itself (the norm weight to ones, the LM head randomly); reading `blocks.{L}.hook_out` never touches them. Two shortcuts give the wrong vector: Hugging Face's last `hidden_states` entry has the final norm applied, and `stop_at_layer=L + 1` does not stop early because the truncated stack has no block $L + 1$, so it runs the full output path. Input that still contains explanation tags is rejected instead of silently stripped, since the AR was trained on the inner text only.

In [7]:
def reconstruct(ar, tokenizer, meta, value_head, text):
    # Predict the activation described by `text` with the AR and its value head.
    assert text.strip(), "need a nonempty explanation"
    assert EXPLANATION_OPEN not in text and EXPLANATION_CLOSE not in text, "pass the inner text"
    layer = meta["extraction_layer_index"]
    assert ar.cfg.n_layers == layer + 1, "the AR must be truncated to layer + 1 blocks"
    rendered = meta["prompt_templates"]["ar"].format(explanation=text)
    ids = tokenizer.encode(rendered, add_special_tokens=True)
    suffix = meta["tokens"]["critic_suffix_ids"]
    assert ids[-len(suffix) :] == suffix, "the explanation changed the prompt suffix"
    assert len(ids) <= ar.cfg.n_ctx, "explanation exceeds the AR context"
    hidden, _ = extract(ar, torch.tensor([ids]), layer, position=-1)
    with torch.no_grad():
        return value_head(hidden.to(value_head.weight.dtype)).float()

## Scoring

Both vectors are rescaled to norm $\sqrt{d}$, matching the sidecar's `mse_scale`, and compared in float64:

$$\cos(h, \hat h) = \frac{h \cdot \hat h}{\lVert h\rVert\,\lVert \hat h\rVert}, \qquad \mathrm{nMSE}(h, \hat h) = \frac{1}{d}\left\lVert \sqrt{d}\,\frac{h}{\lVert h\rVert} - \sqrt{d}\,\frac{\hat h}{\lVert \hat h\rVert}\right\rVert^2 = 2\,(1 - \cos).$$

`score` asserts that identity. These are directional diagnostics for one vector: they are not the raw reconstruction error, and they are not a dataset-level fraction of variance explained.

In [8]:
def score(original, reconstructed):
    # Cosine similarity and direction-normalized MSE of two activation vectors.
    a, b = original.double().flatten(), reconstructed.double().flatten()
    assert a.shape == b.shape, "vectors must have the same width"
    assert torch.isfinite(a).all() and torch.isfinite(b).all() and a.norm() > 0 and b.norm() > 0
    scale = math.sqrt(a.numel())
    a_hat, b_hat = a / a.norm() * scale, b / b.norm() * scale
    # Rounding can push the ratio slightly outside [-1, 1] for (anti)parallel vectors.
    cosine = min(1.0, max(-1.0, float(torch.dot(a, b) / (a.norm() * b.norm()))))
    normalized_mse = float(((a_hat - b_hat) ** 2).mean())
    assert math.isclose(normalized_mse, 2 * (1 - cosine), rel_tol=1e-9, abs_tol=1e-12)
    return {"cosine": cosine, "normalized_mse": normalized_mse}

## Checking the mechanics on a tiny model

The released models are three 7B checkpoints, too large for notebook CI. This section runs every helper above on the two-block `trl-internal-testing/tiny-Qwen2ForCausalLM-2.5` checkpoint, which shares Qwen2.5's architecture and tokenizer, and compares extraction, embedding injection, and the reconstructor read-out against plain Hugging Face, and cached against uncached generation. The tiny model was never trained as an NLA, so its generated text is meaningless and its scores carry no information; only the assertions matter.

The sidecars below are synthetic but follow the released schema. The verbalizer and target share the tiny checkpoint, and the reconstructor is the same checkpoint truncated to $L + 1 = 1$ block with a seeded random value head. Each Hugging Face reference is loaded separately with eager attention, because `boot_transformers(hf_model=...)` routes the wrapped module's own forward pass through the Bridge's hooks.

In [9]:
TINY = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
TINY_LAYER = 0
tiny_tokenizer = AutoTokenizer.from_pretrained(TINY)


def load_hf(name, **kwargs):
    model = AutoModelForCausalLM.from_pretrained(
        name, attn_implementation="eager", dtype=torch.float32, **kwargs
    )
    return model.to(DEVICE).eval()


tiny_target = TransformerBridge.boot_transformers(TINY, device=DEVICE, dtype=torch.float32)
tiny_target.eval()
tiny_av = TransformerBridge.boot_transformers(TINY, device=DEVICE, dtype=torch.float32)
tiny_av.eval()
tiny_ar = TransformerBridge.boot_transformers(
    TINY, hf_model=load_hf(TINY, num_hidden_layers=TINY_LAYER + 1), device=DEVICE
)
tiny_ar.eval()
hf_reference = load_hf(TINY)
TINY_D_MODEL = tiny_target.cfg.d_model

print(f"Tiny target and AV: {tiny_target.cfg.n_layers} blocks, d_model={TINY_D_MODEL}.")
print(f"Tiny AR: {tiny_ar.cfg.n_layers} block (truncated after block {TINY_LAYER}).")

Tiny target and AV: 2 blocks, d_model=8.
Tiny AR: 1 block (truncated after block 0).


In [10]:
def synthetic_sidecar(role, tokenizer, d_model, layer, av_template, ar_template, char):
    # Build a sidecar with the released schema, deriving token ids from the live tokenizer.
    (char_id,) = tokenizer.encode(char, add_special_tokens=False)
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": av_template.format(injection_char=char)}],
        tokenize=True,
        add_generation_prompt=True,
    )
    ids = list(rendered["input_ids"] if hasattr(rendered, "keys") else rendered)
    position = ids.index(char_id)
    # The recorded suffix is the token tail every explanation shares: punctuation at the end of
    # an explanation can merge with the start of the closing tag.
    probes = [
        tokenizer.encode(ar_template.format(explanation=end), add_special_tokens=True)
        for end in ("x", "x.", "x!", "x)")
    ]
    length = 0
    while length < min(map(len, probes)) and len({p[-length - 1] for p in probes}) == 1:
        length += 1
    suffix = probes[0][-length:]
    return {
        "kind": "nla_model",
        "schema_version": 2,
        "role": role,
        "d_model": d_model,
        "extraction_layer_index": layer,
        "extraction": {
            "injection_scale": 150.0 if role == "av" else None,
            "mse_scale": math.sqrt(d_model),
        },
        "tokens": {
            "injection_char": char,
            "injection_token_id": char_id,
            "injection_left_neighbor_id": ids[position - 1],
            "injection_right_neighbor_id": ids[position + 1],
            "critic_suffix_ids": suffix if role == "ar" else None,
        },
        "prompt_templates": {"av": av_template, "ar": ar_template},
    }


SYNTHETIC_TEMPLATES = {
    "av_template": "Explain this concept:\n{injection_char}\nAnswer inside explanation tags.",
    "ar_template": "Summary of the following text: <text>{explanation}</text> <summary>",
    "char": "\u2605",
}
tiny_meta = {
    role: check_sidecar(
        synthetic_sidecar(role, tiny_tokenizer, TINY_D_MODEL, TINY_LAYER, **SYNTHETIC_TEMPLATES),
        role,
        tiny_tokenizer,
        TINY_D_MODEL,
        TINY_LAYER,
    )
    for role in ("av", "ar")
}
torch.manual_seed(0)
tiny_value_head = torch.nn.Linear(TINY_D_MODEL, TINY_D_MODEL, bias=False).eval()

tiny_prompt = tiny_tokenizer("The auditor asked about reasoning.", return_tensors="pt").input_ids
av_prompt, av_position = render_av_prompt(tiny_tokenizer, tiny_meta["av"])
print("Both synthetic sidecars pass check_sidecar.")
print(f"AV prompt: {av_prompt.shape[1]} tokens, placeholder at position {av_position}.")

Both synthetic sidecars pass check_sidecar.
AV prompt: 41 tokens, placeholder at position 29.


### (a) Extraction and injection match Hugging Face

`extract` must return Hugging Face's `hidden_states[L + 1]` at the chosen position, and the AV's prefill logits with the injection hook must equal Hugging Face logits computed from `inputs_embeds` with the same row replaced. Both sides use the same rescaling formula and cast, so any difference would come from the hook placement itself.

In [11]:
tiny_vector, resolved = extract(tiny_target, tiny_prompt, TINY_LAYER, position=-1)
with torch.no_grad():
    hf_hidden = hf_reference(tiny_prompt, output_hidden_states=True).hidden_states
print(f"Resolved position -1 to {resolved} of {tiny_prompt.shape[1]} tokens.")
extract_matches = torch.equal(tiny_vector, hf_hidden[TINY_LAYER + 1][0, -1])
print(f"extract equals HF hidden_states[L + 1]: {extract_matches}")

norm = tiny_meta["av"]["extraction"]["injection_scale"]
hook, calls = make_injection_hook(tiny_vector, av_position, norm, use_past_kv_cache=True)
with torch.no_grad(), tiny_av.hooks(fwd_hooks=[("hook_embed", hook)]):
    bridge_logits = tiny_av(av_prompt)
with torch.no_grad():
    embeddings = hf_reference.get_input_embeddings()(av_prompt)
    injected = embeddings.clone()
    injected[:, av_position] = (tiny_vector.double() / tiny_vector.double().norm() * norm).float()
    hf_logits = hf_reference(inputs_embeds=injected).logits
changed_rows = (injected != embeddings).any(-1)[0].nonzero().flatten().tolist()
print(f"Only the placeholder row is replaced: {changed_rows == [av_position]}")
logits_match = torch.equal(bridge_logits, hf_logits)
print(f"Bridge prefill logits equal HF inputs_embeds logits: {logits_match}")

Resolved position -1 to 5 of 6 tokens.
extract equals HF hidden_states[L + 1]: True
Only the placeholder row is replaced: True
Bridge prefill logits equal HF inputs_embeds logits: True


### (b) Cached and uncached generation agree

Greedy generation with and without the KV cache must produce the same tokens. The hook's call log shows the two injection patterns: one injecting prefill followed by one-token decode steps with the cache, and an injecting full-prefix pass at every step without it.

In [12]:
runs = {
    cached: verbalize(
        tiny_av,
        tiny_tokenizer,
        tiny_meta["av"],
        tiny_vector,
        max_new_tokens=6,
        do_sample=False,
        use_past_kv_cache=cached,
    )
    for cached in (True, False)
}
prompt_len = av_prompt.shape[1]
cached_calls, uncached_calls = runs[True]["hook_calls"], runs[False]["hook_calls"]
tokens_agree = runs[True]["token_ids"] == runs[False]["token_ids"]
print(f"Greedy tokens agree with and without the KV cache: {tokens_agree}")
print(
    "Cached: one injecting prefill, then non-injecting one-token steps: "
    f"{cached_calls[0] == (prompt_len, True) and all(c == (1, False) for c in cached_calls[1:])}"
)
print(
    "Uncached: every pass covers the growing prefix and injects: "
    f"{all(c == (prompt_len + i, True) for i, c in enumerate(uncached_calls))}"
)
print(f"Tiny-model parse status (meaningless text is expected): {runs[True]['status']}")

Greedy tokens agree with and without the KV cache: True
Cached: one injecting prefill, then non-injecting one-token steps: True
Uncached: every pass covers the growing prefix and injects: True
Tiny-model parse status (meaningless text is expected): missing


### (c) The truncated reconstructor reads the raw block output

The AR's block-$L$ output at the final token must equal the full model's `hidden_states[L + 1]`, because the truncated checkpoint shares those blocks. The truncated model's own last `hidden_states` entry differs: Hugging Face applies the final norm to it. `stop_at_layer=L + 1` also fails to stop early on a truncated stack and returns full logits. `reconstruct` must equal the value head applied to the raw read-out.

In [13]:
canonical = "A question about how an auditor reasons."
rendered = tiny_meta["ar"]["prompt_templates"]["ar"].format(explanation=canonical)
ar_tokens = torch.tensor([tiny_tokenizer.encode(rendered, add_special_tokens=True)])
ar_hidden, _ = extract(tiny_ar, ar_tokens, TINY_LAYER)
truncated_hf = load_hf(TINY, num_hidden_layers=TINY_LAYER + 1)
with torch.no_grad():
    full_hidden = hf_reference(ar_tokens, output_hidden_states=True).hidden_states
    truncated_last = truncated_hf(ar_tokens, output_hidden_states=True).hidden_states[-1]
    stopped = tiny_ar(ar_tokens, stop_at_layer=TINY_LAYER + 1)
readout_matches = torch.equal(ar_hidden, full_hidden[TINY_LAYER + 1][0, -1])
normed_differs = not torch.equal(ar_hidden, truncated_last[0, -1])
print(f"AR read-out equals full-model hidden_states[L + 1]: {readout_matches}")
print(f"Truncated HF last hidden_states is normed, so it differs: {normed_differs}")
print(f"stop_at_layer=L + 1 returns vocabulary logits of width {stopped.shape[-1]}, not d_model.")

reconstruction = reconstruct(tiny_ar, tiny_tokenizer, tiny_meta["ar"], tiny_value_head, canonical)
with torch.no_grad():
    expected = tiny_value_head(ar_hidden)
print(f"reconstruct equals value_head(read-out): {torch.equal(reconstruction, expected)}")
del truncated_hf

AR read-out equals full-model hidden_states[L + 1]: True
Truncated HF last hidden_states is normed, so it differs: True
stop_at_layer=L + 1 returns vocabulary logits of width 152064, not d_model.
reconstruct equals value_head(read-out): True


### (d) Explanation parsing

Every parse outcome on a fixed set of continuations, including the tagged-input rejection in `reconstruct`.

In [14]:
parse_cases = [
    ("<explanation> A legal audit. </explanation>", "ok"),
    ("No tags at all.", "missing"),
    ("<explanation>a</explanation> and <explanation>b</explanation>", "multiple"),
    ("<explanation>a <explanation>b</explanation></explanation>", "nested"),
    ("<explanation> cut off by the token budget", "incomplete"),
    ("</explanation> closed before opening <explanation>", "incomplete"),
    ("<explanation>   </explanation>", "empty"),
]
parse_rows = []
for text, expected_status in parse_cases:
    parsed, status = parse_explanation(text)
    assert status == expected_status, (text, status)
    parse_rows.append((text, status, parsed))
display(pd.DataFrame(parse_rows, columns=["continuation", "status", "parsed text"]))

try:
    reconstruct(tiny_ar, tiny_tokenizer, tiny_meta["ar"], tiny_value_head, parse_cases[0][0])
except AssertionError as error:
    print(f"Tagged input rejected by reconstruct: {error}")

,continuation,status,parsed text
0,<explanation> A legal audit. </explanation>,ok,A legal audit.
1,No tags at all.,missing,None
2,<explanation>a</explanation> and <explanation>b</explanation>,multiple,None
3,<explanation>a <explanation>b</explanation></explanation>,nested,None
4,<explanation> cut off by the token budget,incomplete,None
5,</explanation> closed before opening <explanation>,incomplete,None
6,<explanation> </explanation>,empty,None


Tagged input rejected by reconstruct: pass the inner text


### (e) Hooks are removed, including after a failure

After every helper, no hook may remain on any model. To check the failure path, a caller-owned hook that raises is attached to block 0 before `verbalize` runs. The injection hook must be gone after the exception, while the caller's own hook stays until the caller removes it.

In [15]:
def hook_count(bridge):
    # Count torch-level forward hooks on unique hook points (aliases share one HookPoint).
    # The private `_forward_hooks` also sees hooks registered outside the HookPoint API.
    points = {id(point): point for point in bridge.hook_dict.values()}
    return sum(len(point._forward_hooks) for point in points.values())


def failing_hook(activation, hook):
    raise RuntimeError("simulated failure during generation")


hooks_left = [hook_count(m) for m in (tiny_target, tiny_av, tiny_ar)]
print(f"Hooks left after the checks above: {hooks_left}")
caller_point = tiny_av.hook_dict["blocks.0.hook_out"]
caller_point.add_hook(failing_hook)
try:
    verbalize(tiny_av, tiny_tokenizer, tiny_meta["av"], tiny_vector, max_new_tokens=2)
except RuntimeError as error:
    print(f"verbalize raised: {error}")
print(f"Only the caller's hook remains after the failure: {hook_count(tiny_av) == 1}")
caller_point.remove_hooks("fwd")
print(f"No hooks remain once the caller removes it: {hook_count(tiny_av) == 0}")

Hooks left after the checks above: [0, 0, 0]
verbalize raised: simulated failure during generation
Only the caller's hook remains after the failure: True
No hooks remain once the caller removes it: True


### Tiny round trip

The full pipeline end to end. Because the tiny model cannot produce a tagged explanation, the fixed canonical explanation stands in for the verbalizer's output. The score only shows that the pipeline composes; it measures nothing about the tiny model.

In [16]:
tiny_reconstruction = reconstruct(
    tiny_ar, tiny_tokenizer, tiny_meta["ar"], tiny_value_head, canonical
)
tiny_score = score(tiny_vector, tiny_reconstruction)
finite = all(math.isfinite(v) for v in tiny_score.values())
print(f"Score is finite and satisfies nMSE = 2 (1 - cos): {finite}")

del tiny_target, tiny_av, tiny_ar, hf_reference
_ = gc.collect()

Score is finite and satisfies nMSE = 2 (1 - cos): True


## Interpretation checklist

- A high cosine says the AR could recover the activation's direction from the text. It does not say the explanation is what a human would conclude from it, or that the model uses the activation for what the text describes.
- A low cosine can come from a poor explanation, a hard activation, or an AR that ignores parts of the text. One vector cannot separate these.
- Sampled explanations vary between runs. Compare several samples, with fixed seeds, before reading meaning into one.
- Parse failures are outcomes of the AV, not noise to filter. Report them alongside successful round trips.
- The scores are directional. They ignore the activation's norm, and they are not a dataset-level fraction of variance explained.
- An NLA pair is trained for one target model, one layer, and one residual basis. Activations from another model, another layer, or a compatibility-mode Bridge are outside that contract.
- Hooks already installed on a model still apply during these helpers; remove them to analyze the unmodified computation.

## Scope

This notebook covers the inference round trip for one activation at a time. It does not cover steering with edited explanations, corpus-level benchmarks, other released NLA pairs, training, or quantized execution.

## References

- [Natural Language Autoencoders Produce Unsupervised Explanations of LLM Activations](https://transformer-circuits.pub/2026/nla/).
- [Authors' reference implementation](https://github.com/kitft/natural_language_autoencoders). This notebook does not import, vendor, or depend on that repository's code.
- [Patchscopes generation demo](https://github.com/TransformerLensOrg/TransformerLens/blob/dev/demos/Patchscopes_Generation_Demo.ipynb), a related activation-patching technique without a trained reconstructor.